# AI Agent Workflow for UnitCompute

The reusable application code lives in `agent/`. This notebook imports that code so it can be explored interactively.

In [ ]:
from pathlib import Path
import sys

# Jupyter starts from the project root when launched with `jupyter lab`.
sys.path.insert(0, str(Path.cwd()))

from agent.config import load_local_environment

load_local_environment()
print('Local environment loaded.')

In [ ]:
# Provider configuration: replace only this cell to switch providers.
# Examples: ChatGoogleGenerativeAI, ChatOpenAI, ChatBedrockConverse, or another LangChain chat model.
import os

from langchain_google_genai import ChatGoogleGenerativeAI

chat_model = ChatGoogleGenerativeAI(
    model='gemini-2.5-flash',
    google_api_key=os.environ['GOOGLE_API_KEY'],
    temperature=0,
)

## Step 1

Take the user's request and translate it into a format the AI agent can use.

In [ ]:
from langgraph.types import Command

from agent.mappings import (
    COMPUTE_SIZE_TO_INSTANCE_TYPE,
    OPERATING_SYSTEM_TO_IMAGE_PARAMETER,
)
from agent.agent_graph import build_agent_graph
from agent.checkpointing import create_development_checkpointer
from agent.intent_router import IntentRouter
from agent.models import AgentIntent, ResolvedVMPlan, VMPlan, VMRequest
from agent.planner import PlanBuilder
from agent.requirements import RequirementChecker
from agent.translator import RequestTranslator
from agent.workflow import build_create_ec2_workflow

## What is needed for future steps

Decide which values are optional or required, supported instance types, and related policies. The workflow asks follow-up questions for required user information.

In [ ]:
# Preview the user-friendly-to-AWS mappings used by PlanBuilder.
{
    'operating_system': OPERATING_SYSTEM_TO_IMAGE_PARAMETER,
    'compute_size': COMPUTE_SIZE_TO_INSTANCE_TYPE,
}

## Step 2: Build an AWS-ready plan

The user may override optional defaults. Required request details are checked before the plan is built.

In [ ]:
# Example: the user overrides the 30 GB default with 50 GB.
example_request = VMRequest(
    operating_system='ubuntu',
    compute_size='small',
    region='us-east-1',
    disk_gb=50,
)

example_plan = PlanBuilder().build(example_request)
example_plan.model_dump()

## Step 3: Collect missing details and request approval

The workflow asks for one missing required field at a time. Once it has a plan, it returns a review for the user to approve or revise.

In [ ]:
translator = RequestTranslator(chat_model)
intent_router = IntentRouter(chat_model)
checker = RequirementChecker()
plan_builder = PlanBuilder()
create_ec2_workflow = build_create_ec2_workflow(
    translator,
    checker,
    plan_builder,
)
agent_graph = build_agent_graph(
    intent_router,
    create_ec2_workflow,
    checkpointer=create_development_checkpointer(),
)
conversation = {
    'configurable': {'thread_id': 'notebook-demo'},
}

# First message: the Region is missing, so the graph returns a question.
first_result = agent_graph.invoke({
    'user_request': 'I need a small Ubuntu VM for web testing.',
}, conversation)
first_result['__interrupt__']

# Resume the paused graph. Its thread_id loads the saved Ubuntu/small state.
second_result = agent_graph.invoke(
    Command(resume='Use us-east-1.'),
    conversation,
)
second_result['__interrupt__']

# Resume the approval interrupt with an explicit user decision.
approved_result = agent_graph.invoke(
    Command(resume={'action': 'approve'}),
    conversation,
)

approved_plan = approved_result['vm_plan']
approved_plan.model_dump()

## Note for the planning step

The values such as `default-subnet-for-selected-region` should not be hard-coded strings. In the real implementation, the backend will look up the account's default VPC/subnet and ensure the default security group exists. The user can later choose a specific network/security group through the UI if we decide to expose those advanced options.